In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('period_1_train_data.csv')
test = pd.read_csv('test_x.csv')
test_ids = test['id'].copy()

train = train.drop_duplicates()

y_train = train['price_target'].copy()
X_train = train.drop(columns=['price_target'])
X_test = test.drop(columns=['id'])

q_low, q_high = y_train.quantile([0.005, 0.995])
mask_price = (y_train >= q_low) & (y_train <= q_high)
mask_square = (X_train['square'] >= 10) & (X_train['square'] <= 250)
mask_floor = (X_train['floor'] >= -2) & (X_train['floor'] <= 50)

final_mask = mask_price & mask_square & mask_floor
X_train = X_train[final_mask].reset_index(drop=True)
y_train = y_train[final_mask].reset_index(drop=True)

y_train_log = np.log1p(y_train)

def create_features(df):
    df = df.copy()
    
    df['transport_score'] = 0
    if 'location_bus_station_cnt' in df.columns:
        df['transport_score'] += df['location_bus_station_cnt'].fillna(0) * 3
    if 'location_railway_station_cnt' in df.columns:
        df['transport_score'] += df['location_railway_station_cnt'].fillna(0) * 5
    if 'location_public_transport_stop_position_cnt' in df.columns:
        df['transport_score'] += df['location_public_transport_stop_position_cnt'].fillna(0) * 2
    if 'location_railway_station_w_mean_distance' in df.columns:
        df['transport_score'] += df['location_railway_station_w_mean_distance'].fillna(1000) * -0.5
    
    df['infra_score'] = 0
    if 'location_school_w_mean_distance' in df.columns:
        df['infra_score'] += df['location_school_w_mean_distance'].fillna(1000) * -0.8
    if 'location_amenity_pharmacy_w_mean_distance' in df.columns:
        df['infra_score'] += df['location_amenity_pharmacy_w_mean_distance'].fillna(1000) * -0.5
    if 'location_college_cnt' in df.columns:
        df['infra_score'] += df['location_college_cnt'].fillna(0) * 2
    if 'location_marketplace_cnt' in df.columns:
        df['infra_score'] += df['location_marketplace_cnt'].fillna(0) * 1.5
    
    df['comfort_score'] = 0
    if 'location_pop_cafe_cnt' in df.columns:
        df['comfort_score'] += df['location_pop_cafe_cnt'].fillna(0) * 2
    if 'location_leisure_cnt' in df.columns:
        df['comfort_score'] += df['location_leisure_cnt'].fillna(0) * 1.5
    if 'location_water_cnt' in df.columns:
        df['comfort_score'] += df['location_water_cnt'].fillna(0) * 3
    if 'location_parking_cnt' in df.columns:
        df['comfort_score'] += df['location_parking_cnt'].fillna(0) * 2
    
    df['negative_score'] = 0
    if 'location_industrial_cnt' in df.columns:
        df['negative_score'] += df['location_industrial_cnt'].fillna(0) * -3
    if 'location_fuel_cnt' in df.columns:
        df['negative_score'] += df['location_fuel_cnt'].fillna(0) * -2
    if 'location_railway_cnt' in df.columns:
        df['negative_score'] += df['location_railway_cnt'].fillna(0) * -1.5
    
    df['quality_index'] = (
        df['transport_score'].fillna(0) * 0.3 +
        df['infra_score'].fillna(0) * 0.3 +
        df['comfort_score'].fillna(0) * 0.3 +
        df['negative_score'].fillna(0) * 0.1
    )
    
    if 'class_cat' in df.columns:
        df['is_elite'] = (df['class_cat'] == 2581).astype(int)
        df['elite_quality'] = df['is_elite'] * df['quality_index']
    
    if 'floor' in df.columns:
        floor = df['floor'].copy()
        df['floor_type'] = 0
        df.loc[floor < 0, 'floor_type'] = -1
        df.loc[floor == 1, 'floor_type'] = 1
        df.loc[(floor >= 2) & (floor <= 5), 'floor_type'] = 2
        df.loc[(floor >= 6) & (floor <= 10), 'floor_type'] = 3
        df.loc[(floor >= 11) & (floor <= 15), 'floor_type'] = 4
        df.loc[(floor >= 16) & (floor <= 25), 'floor_type'] = 5
        df.loc[floor > 25, 'floor_type'] = 6
        df['floor'] = df['floor'].fillna(5).clip(-1, 50)
        df['floor_anomaly'] = ((floor < 0) | (floor > 30)).astype(int)
    
    if 'square' in df.columns:
        df['square_log'] = np.log1p(df['square'])
        df['square_sqrt'] = np.sqrt(df['square'])
    
    if 'agreement_date' in df.columns:
        dates = pd.to_datetime(df['agreement_date'])
        df['month'] = dates.dt.month
        df['is_expensive_month'] = dates.dt.month.isin([1, 9, 10, 11]).astype(int)
        df['is_cheap_month'] = dates.dt.month.isin([3, 4, 5, 6]).astype(int)
        df['year'] = dates.dt.year
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['quarter'] = dates.dt.quarter
        df['days_from_start'] = (dates - pd.Timestamp('2012-01-01')).dt.days
        df.drop(columns=['agreement_date'], inplace=True)
    
    if 'rooms_4' in df.columns:
        rooms_map = {'studio': 0.5, '1': 1, '2': 2, '3': 3, '4': 4, '5+': 5.5}
        df['rooms_num'] = df['rooms_4'].map(rooms_map).fillna(2)
        df['area_per_room'] = df['square'] / (df['rooms_num'] + 0.5)
        rooms_dummies = pd.get_dummies(df['rooms_4'], prefix='rooms', drop_first=True).astype(int)
        df = pd.concat([df, rooms_dummies], axis=1)
        df.drop(columns=['rooms_4'], inplace=True)
    
    if 'class_cat' in df.columns:
        class_order = {97865: 0, 27353: 1, 2581: 2}
        df['class_encoded'] = df['class_cat'].map(class_order).fillna(0).astype(int)
        df.drop(columns=['class_cat'], inplace=True)
    
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    
    return df.fillna(df.median())

X_train = create_features(X_train)
X_test = create_features(X_test)

distance_cols = [c for c in X_train.columns if 'w_mean_distance' in c]
count_cols = [c for c in X_train.columns if c.endswith('_cnt')]

if distance_cols:
    X_train['loc_dist_mean'] = X_train[distance_cols].mean(axis=1)
    X_train['loc_dist_std'] = X_train[distance_cols].std(axis=1).fillna(0)
    X_test['loc_dist_mean'] = X_test[distance_cols].mean(axis=1)
    X_test['loc_dist_std'] = X_test[distance_cols].std(axis=1).fillna(0)

if count_cols:
    X_train['loc_cnt_sum'] = X_train[count_cols].sum(axis=1)
    X_test['loc_cnt_sum'] = X_test[count_cols].sum(axis=1)

loc_features = [c for c in X_train.columns if c.startswith('location_') and 
                'dist_' not in c and 'cnt_' not in c and '_exists' not in c]

if len(loc_features) > 20:
    X_train_loc = X_train[loc_features].fillna(X_train[loc_features].median())
    X_test_loc = X_test[loc_features].fillna(X_train[loc_features].median())
    
    pca = PCA(n_components=30, random_state=42)
    X_train_pca = pca.fit_transform(X_train_loc)
    X_test_pca = pca.transform(X_test_loc)
    
    for i in range(30):
        X_train[f'pca_{i}'] = X_train_pca[:, i]
        X_test[f'pca_{i}'] = X_test_pca[:, i]
    
    X_train = X_train.drop(columns=loc_features)
    X_test = X_test.drop(columns=loc_features)

X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())
X_test = X_test[X_train.columns]

params = {
    'objective': 'regression',
    'metric': 'mape',
    'num_leaves': 150,
    'learning_rate': 0.02,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 3,
    'n_estimators': 3000,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
val_scores = []
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_log.iloc[train_idx], y_train.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr)
    pred = np.expm1(model.predict(X_val))
    mape = mean_absolute_percentage_error(y_val, pred) * 100
    val_scores.append(mape)
    print(f"  Fold {fold+1}: {mape:.4f}%")
    test_preds += np.expm1(model.predict(X_test)) / n_folds

print(f"\nИтоговый MAPE: {np.mean(val_scores):.4f}%")

final_pred = np.maximum(test_preds, 10000)
submission = pd.DataFrame({'id': test_ids, 'price_target': final_pred})
submission.to_csv('submission_3000_trees.csv', index=False)

  Fold 1: 1.9626%
  Fold 2: 1.9757%
  Fold 3: 1.9613%
  Fold 4: 2.0381%
  Fold 5: 1.9451%

Итоговый MAPE: 1.9766%
